# Part 2: Simple Logistic Regression Model
We will be using a simple bag-of-words representation for our initial, simple model. To train this model, we will be using the pytorch library.

In [1]:
from modelling import BagofWordsDataset, train
from torch.utils.data import DataLoader
from pathlib import Path
import torch.nn as nn
import torch

In [ ]:
DATASET_DIRECTORY = Path("data/final_dataset")
VOCABULARY = DATASET_DIRECTORY / "top10k_vocabulary.csv"
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
EPOCHS = 10
L2_LAMBDA = 0.01 # add l2 reg to discourage the model from giving giant weights to extremely rare tokens

# To stop using domain: remove "include_domain=True and unique_domains=..." for both BagOfWordsDatasets
train_ds = BagofWordsDataset(DATASET_DIRECTORY / "train.csv", VOCABULARY, include_domain=True)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE)
val_dl = DataLoader(BagofWordsDataset(DATASET_DIRECTORY / "val.csv", VOCABULARY, include_domain=True, unique_domains=train_ds.__get_unique_domains__()), batch_size=BATCH_SIZE)

tokenizing 634696 documents


100%|██████████| 634696/634696 [04:06<00:00, 2573.71it/s]


tokenizing 79337 documents


100%|██████████| 79337/79337 [00:31<00:00, 2556.19it/s]


In [5]:
from modelling.logistic_regression import LogisticRegression
model = LogisticRegression(len(train_ds.__getitem__(0)[0]))
loss = nn.BCELoss()

train(model, loss, train_dl, val_dl, LEARNING_RATE, EPOCHS)

100%|██████████| 1240/1240 [00:05<00:00, 227.68it/s]


Epochs: 1 | Train Loss:  0.378 | Val Loss:  0.277 
[+] Saved checkpoint 'best_model_2026-03-14_19:49_epoch_0.pth'


100%|██████████| 1240/1240 [00:05<00:00, 209.13it/s]


Epochs: 2 | Train Loss:  0.223 | Val Loss:  0.187 
[+] Saved checkpoint 'best_model_2026-03-14_19:49_epoch_1.pth'


100%|██████████| 1240/1240 [00:05<00:00, 216.76it/s]


...

Epochs: 8 | Train Loss:  0.041 | Val Loss:  0.042 
[+] Saved checkpoint 'best_model_2026-03-14_19:49_epoch_7.pth'


100%|██████████| 1240/1240 [00:05<00:00, 234.99it/s]


Epochs: 9 | Train Loss:  0.034 | Val Loss:  0.036 
[+] Saved checkpoint 'best_model_2026-03-14_19:49_epoch_8.pth'


100%|██████████| 1240/1240 [00:05<00:00, 220.45it/s]

Epochs: 10 | Train Loss:  0.028 | Val Loss:  0.031 
[+] Saved checkpoint 'best_model_2026-03-14_19:49_epoch_9.pth'


In [ ]:
from sklearn.metrics import f1_score
from tqdm import tqdm

outputs = []
labels = []

for inputs, b_labels in tqdm(val_dl):
    labels = [*labels, *b_labels]
    probabilities = model(inputs)
    classes = (probabilities >= 0.5).long()
    outputs = [*outputs, *classes.tolist()]

print('F1-Score: ',f1_score(outputs, labels))

# without domain: 0.888
# with domain: 0.997

100%|██████████| 1240/1240 [00:05<00:00, 211.70it/s]


F1-Score:  0.9968992095445284


In [7]:
acc = torch.tensor([1 if labels[i].item() == outputs[i] else 0 for i in range(len(outputs))])
acc.mean(dtype=float)

tensor(0.9960, dtype=torch.float64)